In [1]:
import pandas as pd

base = pd.read_excel(r'data/Base_Atendimentos_v3_BRUTA.xlsx')
base['Status_Atendimento'].value_counts(dropna=False)
display(base[:5])

base.info()

,ID_Atendimento,Data,Colaborador,Fila_Atendimento,Status_Atendimento,Avaliado,Nota_Avaliacao,TMA,TME,Mes,Ano,Canal,Assunto,Cooperativa,Hora_Inicio,Resolvido_1o_Contato
0,105492,02/07/2026,lucas,Recompra de Cotas Excluídas,Encerrado por Inatividade,Não,NaN,2:27:37,00:44:56,7,2026,Chat,Dúvida sobre valor,Singular 001,13:53:44,NaN
1,137525,10/03/2026,Aline,Atendimento Geral,Concluído,Não,NaN,01:42:36,00:23:26,3,2026,Chat,Solicitação de extrato,Singular 005,16:26:26,Sim
2,106338,26/08/2026,Lucas,Pós Contemplação,Concluído,Não,NaN,02:32:56,00:19:09,8,2026,Chat,Seguro,Singular 005,08:14:46,Sim
3,127523,27/11/2026,Sandra,Cancelamento/Retenção,Abandono do Cliente,Não,NaN,02:59:43,00:10:43,11,2026,Telefone,Proposta de retenção,Singular 004,14:17:33,NaN
4,134333,01/10/2026,Wilker,Recompra de Cotas Excluídas,Abandono do Cliente,Não,NaN,:41:13,00:37:02,10,2026,WhatsApp,Prazo de pagamento,Singular 005,16:56:12,NaN


<class 'pandas.DataFrame'>
RangeIndex: 43487 entries, 0 to 43486
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ID_Atendimento        43487 non-null  int64 
 1   Data                  43487 non-null  str   
 2   Colaborador           43487 non-null  str   
 3   Fila_Atendimento      43487 non-null  str   
 4   Status_Atendimento    43487 non-null  str   
 5   Avaliado              43487 non-null  str   
 6   Nota_Avaliacao        16382 non-null  object
 7   TMA                   43487 non-null  str   
 8   TME                   43487 non-null  str   
 9   Mes                   43487 non-null  int64 
 10  Ano                   43487 non-null  int64 
 11  Canal                 43487 non-null  str   
 12  Assunto               43487 non-null  str   
 13  Cooperativa           43487 non-null  str   
 14  Hora_Inicio           43487 non-null  str   
 15  Resolvido_1o_Contato  32126 non-null  str   
dt

In [2]:
def limpar_texto(coluna):
    return(
    coluna.astype(str)
    .str.upper()
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
    .str.strip()
)
# tratando e limpando as colunas e depois renomeando com replace 
colunas_p_limpar = ['Colaborador', 'Fila_Atendimento', 'Status_Atendimento', 'Avaliado']
base[colunas_p_limpar] = base[colunas_p_limpar].apply(limpar_texto)

base['Nota_Avaliacao'] = (base['Nota_Avaliacao'].astype(str).str.replace(',', '.', regex=False))
base['Nota_Avaliacao'] = pd.to_numeric(base['Nota_Avaliacao'], errors='coerce')
base = base[~base['Colaborador'].str.contains('TESTE', na=False)]

base['Colaborador'] = base['Colaborador'].replace({'LUCAS': 'Lucas', 'ALINE': 'Aline', 'WILKER': 'Wilker', 'VINICIUS': 'Vinícius', 'SANDRA': 'Sandra'})
base['Avaliado'] = base['Avaliado'].replace({'N': 'Não', 'S': 'Sim', 'n': 'Não', 's': 'Sim', 'NAO': 'Não', 'SIM': 'Sim'})
nome_filas = {'RECOMPRA DE COTAS EXCLUIDAS': 'Recompra de Cotas Excluídas', 'CANCELAMENTO/RETENCAO': 'Cancelamento/Retenção', 'ATENDIMENTO GERAL': 'Atendimento Geral', 'POS CONTEMPLACAO': 'Pós Contemplação'}
base['Fila_Atendimento'] = base['Fila_Atendimento'].map(nome_filas)
nome_status = {'CONCLUIDO': 'Concluído', 'ENCERRADO POR INATIVIDADE': 'Encerrado por Inatividade', 'ABANDONO DO CLIENTE': 'Abandono'}
base['Status_Atendimento'] = base['Status_Atendimento'].replace(nome_status)

# converter data - esqueci hehe, so lembrei do info() agora
# para o Lucas do futuro, são TRÊS formatos 00-00-0000, 00/00/0000 e 0000-00-00, convertemos todos e juntamos com o (fiilna)
data_iso = pd.to_datetime(base['Data'], format = '%Y-%m-%d', errors = 'coerce')
data_br = pd.to_datetime(base['Data'], format = '%d/%m/%Y', errors = 'coerce')
data_br2 = pd.to_datetime(base['Data'], format = '%d-%m-%Y', errors = 'coerce')
base['Data'] = data_iso.fillna(data_br).fillna(data_br2)

## abaixo confere se ainda há data errada
display(base[base['Data'].isna()]['Data'].head(10))

# converter horario TMA, TME e Hora_Inicio / algumas linhas do TMA estão faltando o 0, fiz logo com todas para evitar erros e converter 
def converter_timetodelta(coluna):
    hora_corrigida = (coluna.astype(str).str.strip().str.replace(r'^:', '0:', regex=True))
    return pd.to_timedelta(hora_corrigida, errors = 'coerce')
#converte
base['TMA'] = converter_timetodelta(base['TMA'])
base['TME'] = converter_timetodelta(base['TME'])
base['Hora_Inicio'] = converter_timetodelta(base['Hora_Inicio'])

display(base[:5])
base.info()

Series([], Name: Data, dtype: datetime64[us])

,ID_Atendimento,Data,Colaborador,Fila_Atendimento,Status_Atendimento,Avaliado,Nota_Avaliacao,TMA,TME,Mes,Ano,Canal,Assunto,Cooperativa,Hora_Inicio,Resolvido_1o_Contato
0,105492,2026-07-02,Lucas,Recompra de Cotas Excluídas,Encerrado por Inatividade,Não,NaN,0 days 02:27:37,0 days 00:44:56,7,2026,Chat,Dúvida sobre valor,Singular 001,0 days 13:53:44,NaN
1,137525,2026-03-10,Aline,Atendimento Geral,Concluído,Não,NaN,0 days 01:42:36,0 days 00:23:26,3,2026,Chat,Solicitação de extrato,Singular 005,0 days 16:26:26,Sim
2,106338,2026-08-26,Lucas,Pós Contemplação,Concluído,Não,NaN,0 days 02:32:56,0 days 00:19:09,8,2026,Chat,Seguro,Singular 005,0 days 08:14:46,Sim
3,127523,2026-11-27,Sandra,Cancelamento/Retenção,Abandono,Não,NaN,0 days 02:59:43,0 days 00:10:43,11,2026,Telefone,Proposta de retenção,Singular 004,0 days 14:17:33,NaN
4,134333,2026-10-01,Wilker,Recompra de Cotas Excluídas,Abandono,Não,NaN,0 days 00:41:13,0 days 00:37:02,10,2026,WhatsApp,Prazo de pagamento,Singular 005,0 days 16:56:12,NaN


<class 'pandas.DataFrame'>
Index: 43367 entries, 0 to 43486
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype          
---  ------                --------------  -----          
 0   ID_Atendimento        43367 non-null  int64          
 1   Data                  43367 non-null  datetime64[us] 
 2   Colaborador           43367 non-null  str            
 3   Fila_Atendimento      43367 non-null  str            
 4   Status_Atendimento    43367 non-null  str            
 5   Avaliado              43367 non-null  str            
 6   Nota_Avaliacao        16337 non-null  float64        
 7   TMA                   43367 non-null  timedelta64[us]
 8   TME                   43367 non-null  timedelta64[us]
 9   Mes                   43367 non-null  int64          
 10  Ano                   43367 non-null  int64          
 11  Canal                 43367 non-null  str            
 12  Assunto               43367 non-null  str            
 13  Cooperativa      

In [3]:
#por fim, conferimos as duplicidas e armanezamos o resultado do drop em uma nova base = base2
print(base['ID_Atendimento'].duplicated().sum())
display(base[base['ID_Atendimento'].duplicated(keep=False)].sort_values('ID_Atendimento'))

base2 = base.drop_duplicates(subset=['ID_Atendimento'], keep="first")

641


,ID_Atendimento,Data,Colaborador,Fila_Atendimento,Status_Atendimento,Avaliado,Nota_Avaliacao,TMA,TME,Mes,Ano,Canal,Assunto,Cooperativa,Hora_Inicio,Resolvido_1o_Contato
33400,100006,2026-01-30,Lucas,Atendimento Geral,Concluído,Sim,4.0,0 days 03:03:54,0 days 00:13:08,1,2026,Chat,Dúvida sobre assembleia,Singular 003,0 days 09:12:16,Não
20273,100006,2026-01-30,Lucas,Atendimento Geral,Concluído,Sim,4.0,0 days 03:03:54,0 days 00:13:08,1,2026,Chat,Dúvida sobre assembleia,Singular 003,0 days 09:12:16,Não
12282,100047,2026-01-28,Lucas,Recompra de Cotas Excluídas,Concluído,Sim,4.0,0 days 02:45:04,0 days 00:19:56,1,2026,WhatsApp,Prazo de pagamento,Singular 004,0 days 14:21:12,Sim
23381,100047,2026-01-28,Lucas,Recompra de Cotas Excluídas,Concluído,Sim,4.0,0 days 02:45:04,0 days 00:19:56,1,2026,WhatsApp,Prazo de pagamento,Singular 004,0 days 14:21:12,Sim
2009,100058,2026-01-01,Lucas,Pós Contemplação,Concluído,Sim,2.0,0 days 02:30:21,0 days 00:19:14,1,2026,Chat,Documentação do bem,Singular 007,0 days 09:14:29,Sim
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27203,142528,2026-12-31,Aline,Cancelamento/Retenção,Concluído,Não,NaN,0 days 02:28:05,0 days 00:05:01,12,2026,Telefone,Solicitação de cancelamento,Singular 003,0 days 10:03:37,Sim
7048,142558,2026-12-31,Aline,Atendimento Geral,Concluído,Não,NaN,0 days 01:42:33,0 days 00:03:41,12,2026,WhatsApp,Solicitação de extrato,Singular 003,0 days 11:45:35,Não
15472,142558,2026-12-31,Aline,Atendimento Geral,Concluído,Não,NaN,0 days 01:42:33,0 days 00:03:41,12,2026,WhatsApp,Solicitação de extrato,Singular 003,0 days 11:45:35,Não
33264,142658,2026-12-15,Aline,Atendimento Geral,Encerrado por Inatividade,Não,NaN,0 days 01:35:25,0 days 00:27:41,12,2026,Chat,Solicitação de extrato,Singular 003,0 days 15:45:26,NaN


In [4]:
'''Robo de conferencia da conversão da HORA

quantidade_nat = base['Hora_Inicio'].isna().sum()
print(f'Quantidade de NaT restantes: {quantidade_nat}')'''

'''Robo de conferencia da conversão da data
# FOI O CHAT QUE FEZ
# Passo A: Verificar se sobrou algum NaT (Valor de data nulo/inválido)
quantidade_nat = base['Data_Convertida'].isna().sum()
print(f'Quantidade de NaT restantes: {quantidade_nat}')

# Passo B: Extrair o mês da data convertida para comparar
# O .dt.month extrai o número do mês (1 a 12)
base['Mes_Convertido'] = base['Data_Convertida'].dt.month

# Passo C: Comparar se a coluna 'Mes' original bate 100% com o 'Mes_Convertido'
# (Certifique-se de que a sua coluna 'Mes' original também seja do tipo numérico)
base['Mes'] = pd.to_numeric(base['Mes'], errors='coerce')

# Cria uma coluna True/False para verificar a igualdade linha por linha
base['Meses_Batem'] = base['Mes_Convertido'] == base['Mes']

# Calcula a porcentagem de acerto
porcentagem_acerto = base['Meses_Batem'].mean() * 100
print(f'Porcentagem de meses correspondentes: {porcentagem_acerto:.2f}%')

# Validação final em texto
if quantidade_nat == 0 and porcentagem_acerto == 100.0:
  print('✅ Sucesso! Conversão 100% segura e validada.')
else:
  print('❌ Alerta! Ainda há divergências nas datas ou meses.')'''

"Robo de conferencia da conversão da data\n# FOI O CHAT QUE FEZ\n# Passo A: Verificar se sobrou algum NaT (Valor de data nulo/inválido)\nquantidade_nat = base['Data_Convertida'].isna().sum()\nprint(f'Quantidade de NaT restantes: {quantidade_nat}')\n\n# Passo B: Extrair o mês da data convertida para comparar\n# O .dt.month extrai o número do mês (1 a 12)\nbase['Mes_Convertido'] = base['Data_Convertida'].dt.month\n\n# Passo C: Comparar se a coluna 'Mes' original bate 100% com o 'Mes_Convertido'\n# (Certifique-se de que a sua coluna 'Mes' original também seja do tipo numérico)\nbase['Mes'] = pd.to_numeric(base['Mes'], errors='coerce')\n\n# Cria uma coluna True/False para verificar a igualdade linha por linha\nbase['Meses_Batem'] = base['Mes_Convertido'] == base['Mes']\n\n# Calcula a porcentagem de acerto\nporcentagem_acerto = base['Meses_Batem'].mean() * 100\nprint(f'Porcentagem de meses correspondentes: {porcentagem_acerto:.2f}%')\n\n# Validação final em texto\nif quantidade_nat == 0 and

In [5]:
base2['Mes_Ano'] = (base2['Data'].dt.strftime('%B de %Y').str.capitalize()
)  # Ex: "Janeiro de 2026"

# 3. Calcula o Bimestre de forma automática (Mês 1 e 2 = Bimestre 1, Mês 3 e 4 = Bimestre 2...)
base2['Bimestre'] = ((base2['Mes'] - 1) // 2) + 1
base2['Bimestre_Nome'] = base2['Bimestre'].apply(lambda x: f'{x}º Bimestre')
#criando um dicionario para corrigir os nomes dos meses e depois transformando o ano em str para juntar ao nome e ter a columa com nome do mes e o ano
meses = {1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril', 5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto', 9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'}
base2['Mes_Ano'] = base2['Mes'].map(meses) + ' de ' + base2['Ano'].astype(str)
display(base2[:2])

,ID_Atendimento,Data,Colaborador,Fila_Atendimento,Status_Atendimento,Avaliado,Nota_Avaliacao,TMA,TME,Mes,Ano,Canal,Assunto,Cooperativa,Hora_Inicio,Resolvido_1o_Contato,Mes_Ano,Bimestre,Bimestre_Nome
0,105492,2026-07-02,Lucas,Recompra de Cotas Excluídas,Encerrado por Inatividade,Não,NaN,0 days 02:27:37,0 days 00:44:56,7,2026,Chat,Dúvida sobre valor,Singular 001,0 days 13:53:44,NaN,Julho de 2026,4,4º Bimestre
1,137525,2026-03-10,Aline,Atendimento Geral,Concluído,Não,NaN,0 days 01:42:36,0 days 00:23:26,3,2026,Chat,Solicitação de extrato,Singular 005,0 days 16:26:26,Sim,Março de 2026,2,2º Bimestre


In [6]:
#conferindo se tudo está correto
print(base2['Fila_Atendimento'].value_counts())
print(base2['Status_Atendimento'].value_counts())
print(base2['Avaliado'].value_counts())
print(base2['Nota_Avaliacao'].value_counts())
print(base2['Nota_Avaliacao'].notna().sum())
base2.info()

Fila_Atendimento
Recompra de Cotas Excluídas    11452
Cancelamento/Retenção          10998
Atendimento Geral              10529
Pós Contemplação                9747
Name: count, dtype: int64
Status_Atendimento
Concluído                    31580
Encerrado por Inatividade     6865
Abandono                      4281
Name: count, dtype: int64
Avaliado
Não    26611
Sim    16115
Name: count, dtype: int64
Nota_Avaliacao
5.0    10063
4.0     3874
3.0     1080
2.0      717
1.0      381
Name: count, dtype: int64
16115
<class 'pandas.DataFrame'>
Index: 42726 entries, 0 to 43486
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype          
---  ------                --------------  -----          
 0   ID_Atendimento        42726 non-null  int64          
 1   Data                  42726 non-null  datetime64[us] 
 2   Colaborador           42726 non-null  str            
 3   Fila_Atendimento      42726 non-null  str            
 4   Status_Atendimento    42726 non-nu

In [7]:
#dataFrame/dicionario para cada colaborador, separado por groupby
dic_colaborador = dict(list(base2.groupby('Colaborador')))
#dia_indicador = base['Data'].max()

print(len(dic_colaborador))
display(dic_colaborador['Lucas'][:3])


5


,ID_Atendimento,Data,Colaborador,Fila_Atendimento,Status_Atendimento,Avaliado,Nota_Avaliacao,TMA,TME,Mes,Ano,Canal,Assunto,Cooperativa,Hora_Inicio,Resolvido_1o_Contato,Mes_Ano,Bimestre,Bimestre_Nome
0,105492,2026-07-02,Lucas,Recompra de Cotas Excluídas,Encerrado por Inatividade,Não,NaN,0 days 02:27:37,0 days 00:44:56,7,2026,Chat,Dúvida sobre valor,Singular 001,0 days 13:53:44,NaN,Julho de 2026,4,4º Bimestre
2,106338,2026-08-26,Lucas,Pós Contemplação,Concluído,Não,NaN,0 days 02:32:56,0 days 00:19:09,8,2026,Chat,Seguro,Singular 005,0 days 08:14:46,Sim,Agosto de 2026,4,4º Bimestre
5,104294,2026-05-06,Lucas,Atendimento Geral,Concluído,Não,NaN,0 days 01:50:37,0 days 00:22:41,5,2026,Telefone,Segunda via de boleto,Singular 002,0 days 08:33:08,Sim,Maio de 2026,3,3º Bimestre


In [8]:
def calcular_isc(df_isc):
    df_qtd_avaliados = df_isc[df_isc['Avaliado'] == 'Sim']
    if len(df_qtd_avaliados) == 0:
        return 0.0 
    df_notas_boas = df_qtd_avaliados['Nota_Avaliacao'].isin([4, 5])
    return df_notas_boas.mean() * 100
# geral equipe
isc_equipe = calcular_isc(base)
print('ISC geral da equipe: {:.2f}%'.format(isc_equipe))
# separado para cada colaborador
for nome, isc_colab in dic_colaborador.items():
    isc_colaborador = calcular_isc(isc_colab)
    print('Colaborador: {} | ISC: {:.2f}%'.format(nome, isc_colaborador))

ISC geral da equipe: 86.48%
Colaborador: Aline | ISC: 84.23%
Colaborador: Lucas | ISC: 88.32%
Colaborador: Sandra | ISC: 85.86%
Colaborador: Vinícius | ISC: 87.73%
Colaborador: Wilker | ISC: 85.01%


In [9]:
isc_por_bimestre = (base2.groupby(['Mes', 'Bimestre_Nome', 'Mes_Ano']).apply(calcular_isc).reset_index(name = 'ISC (%)').round(2))
isc_por_bimestre = isc_por_bimestre.drop(columns = ['Mes'])
relatorio_por_colaborador = (base2.groupby(['Mes', 'Colaborador', 'Bimestre_Nome', 'Mes_Ano']).apply(calcular_isc).reset_index(name = 'ISC (%)').round(2))
relatorio_por_colaborador = relatorio_por_colaborador.drop(columns = ['Mes'])
display(isc_por_bimestre)
display(relatorio_por_colaborador)

,Bimestre_Nome,Mes_Ano,ISC (%)
0,1º Bimestre,Janeiro de 2026,82.65
1,1º Bimestre,Fevereiro de 2026,86.30
2,2º Bimestre,Março de 2026,85.33
3,2º Bimestre,Abril de 2026,89.25
4,3º Bimestre,Maio de 2026,85.91
5,3º Bimestre,Junho de 2026,87.53
6,4º Bimestre,Julho de 2026,88.07
7,4º Bimestre,Agosto de 2026,88.82
8,5º Bimestre,Setembro de 2026,89.40
9,5º Bimestre,Outubro de 2026,82.19


,Colaborador,Bimestre_Nome,Mes_Ano,ISC (%)
0,Aline,1º Bimestre,Janeiro de 2026,77.67
1,Lucas,1º Bimestre,Janeiro de 2026,88.03
2,Sandra,1º Bimestre,Janeiro de 2026,82.46
3,Vinícius,1º Bimestre,Janeiro de 2026,81.48
4,Wilker,1º Bimestre,Janeiro de 2026,80.77
5,Aline,1º Bimestre,Fevereiro de 2026,81.28
6,Lucas,1º Bimestre,Fevereiro de 2026,81.70
7,Sandra,1º Bimestre,Fevereiro de 2026,82.83
8,Vinícius,1º Bimestre,Fevereiro de 2026,93.02
9,Wilker,1º Bimestre,Fevereiro de 2026,89.39


In [10]:
#import pathlib

#caminho_backup = pathlib.Path(r'backup')
#arquivos_backup = caminho_backup.iterdir()
#lista_colaborador_backup = [arquivo.name.strip().upper() for arquivo in arquivos_backup]
#for colaborador in dic_colaborador:
#    colaborador_limpo = colaborador.strip().upper()
#    if colaborador_limpo not in lista_colaborador_backup:
#        nova_pasta = caminho_backup / colaborador_limpo
#        nova_pasta.mkdir(exist_ok=True)